# Link2iOS — The Legend of Zelda: Breath of the Wild

Notebook já configurado para a pasta Google Drive compartilhada com a service account. O próprio Colab baixa o RAR de base + update, valida e extrai os NSPs, e mantém somente os NSPs extraídos no Drive. `prod.keys` é lido diretamente da raiz da pasta. O RAR temporário é apagado.

O processo aceita BOTW base + update **sem DLC**. O conteúdo de expansão não será incluído. O Colab prepara o jogo e o GitHub Actions/macOS finaliza a IPA NX2Apple Route C.

**Antes de executar:** o Secret `LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON` já deve estar configurado no Colab e `GOOGLE_SERVICE_ACCOUNT_JSON` no GitHub Actions. Não mova o `prod.keys` da raiz da pasta do Drive.

In [ ]:
import os
from getpass import getpass
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
from google.colab import userdata
assert userdata.get("LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON"), "Configure o Secret LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON no Colab."
DOWNLOAD_URL = getpass("Cole o link assinado do RAR (na primeira execução; pode deixar vazio se BOTW_Extracted já existe no Drive): ").strip()
if DOWNLOAD_URL:
    os.environ["LINK2IOS_DOWNLOAD_URL"] = DOWNLOAD_URL
else:
    os.environ.pop("LINK2IOS_DOWNLOAD_URL", None)
del DOWNLOAD_URL
print("Drive configurado. O link foi mantido apenas na memória desta sessão e não será gravado no GitHub/Drive.")


## 1. Obter o pipeline desta branch

Este clone lê os scripts da branch `Link2iOS`; nenhuma alteração local é enviada automaticamente.

In [ ]:
%cd /content
!rm -rf /content/Link2iOS-repo
!git clone --depth 1 --branch Link2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/Link2iOS-repo
!git -C /content/Link2iOS-repo rev-parse --short HEAD


## 2. Colab/Linux: download, extração e preparação BOTW

Na primeira execução, o Colab baixa o RAR pelo link assinado, valida o arquivo e extrai seu conteúdo. Procura os NSPs recursivamente e envia somente os `.nsp` extraídos para `BOTW_Extracted/` na pasta compartilhada. O arquivo `.rar` nunca é enviado ao Drive e é apagado do armazenamento temporário do Colab.

O `prod.keys` é obtido da raiz da pasta compartilhada. O pipeline baixa a chave para o runtime temporário, usa-a na descriptografia e a apaga antes de arquivar o workspace. Os NSPs extraídos permanecem no Drive e serão reutilizados nas próximas execuções; não haverá novo download do RAR.

O pacote precisa ser BOTW base + update 1.9.0 (`01007EF00011E000`, `v1114112`). DLC é opcional; esta execução não requer AOC.

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py prepare --drive-folder-id "$DRIVE_FOLDER_ID"


## 3. Última célula: enviar o handoff e disparar somente Link2iOS

A última célula pede seu Fine-grained PAT. Ele precisa estar limitado a este repositório com **Contents: Read and write**. O token é usado apenas para criar/atualizar `handoff.json` na branch `Link2iOS`; ele não é salvo no notebook, Drive ou repositório. Esse commit é o gatilho do único workflow.

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py dispatch --handoff /content/link2ios-handoff.json
